# Wind Energy

In [ ]:
import scipy.stats as sps
from scipy.special import gamma
import numpy as np
import matplotlib.pyplot as plt
import enn554.wind as wind
from enn554.paths import data_dir
dd = data_dir()

## Method of bins

In [ ]:
coopers_gap = wind.merra_wind_speed_data()
coopers_gap.import_data(dd/"tutorial_4/POWER_Point_Hourly_20150101_20241231_026d73S_151d47E_UTC.csv")
coopers_gap.utc_to_lst(10) # we only really need to do this if we are aligning with the load.

fig,ax = plt.subplots()
counts, bin_edges, patches = ax.hist(coopers_gap.data['WS50M'])
m = 0.5 * (bin_edges[1:] + bin_edges[:-1])
ax.scatter(m,np.zeros_like(m),marker='*',color='red')

In [ ]:
u_cutin  = 3.5
u_rated  = 13.5
u_cutout = 25
p_rated  = 1000.0  # kW

def power_curve(u):
    p = np.where(
        (u >= u_cutin) & (u < u_rated),
        p_rated * ((u - u_cutin) / (u_rated - u_cutin)) ** 3,
        np.where(
            (u >= u_rated) & (u <= u_cutout),
            p_rated,
            0.0
        )
    )
    return p

In [ ]:
sorted_ws = coopers_gap.data['WS50M'].values
sorted_ws = np.flip(np.sort(sorted_ws))
hours_above = np.arange(1, len(sorted_ws)+1)
fig, ax = plt.subplots()
ax.plot(8760*hours_above/len(sorted_ws),sorted_ws) # scale x-axis to hours per year
ax.set_xlabel("Hours per year")
ax.set_ylabel("Wind Speed (m/s)")
ax.set_title("Speed duration curve at 100m")


In [ ]:
ws = coopers_gap.data['WS50M'].values
turbine_power = power_curve(sorted_ws)
hours_above = np.arange(1, len(turbine_power)+1)
fig, ax = plt.subplots()
ax.plot(8760*hours_above/len(turbine_power),turbine_power) # scale x-axis to hours per year
ax.set_xlabel("Hours per year")
ax.set_ylabel("Wind Power (kW)")
ax.set_title("Power duration curve")

## Testing the average power formula

In [ ]:
c,k = 100,2.0
ρ=1.22
N = 1000000
u = sps.weibull_min(k,scale=c).rvs(N)
WPD = 0.5*ρ*(u**3).mean()
WPD_analytical = 0.5*ρ*c**3*gamma(1+3.0/k)
print(f'Monte Carlo: {WPD:.2f}, Analytical: {WPD_analytical:.2f}')
print(f'Representative: {0.5*ρ*u.mean()**3:.2f}')

In [ ]:
# Power curve parameters
plt.rcParams.update({
    'font.size':        14,   # default for all text
    'axes.titlesize':   16,   # axes title
    'axes.labelsize':   14,   # x and y labels
    'xtick.labelsize':  12,   # tick labels
    'ytick.labelsize':  12,
    'legend.fontsize':  12,
    'figure.titlesize': 18,   # suptitle
})


u = np.linspace(0, 27, 500)
p = power_curve(u)

fig, ax = plt.subplots(figsize=(8, 5))
ax2 = ax.twinx()
ax.plot(u, p, color='gray', linewidth=2,label='Power curve')

ax.set_xlabel('Wind speed (m/s)')
ax.set_ylabel(r'$P_w$')
ax.set_xlim(0, 27)

ax.set_ylim(-100, 1500)
ax.set_xticks([5, 10, 15, 20, 25])
ax.set_yticks([0, 250, 500, 750, 1000, 1250])
ax.set_facecolor('#e8e8e8')
ax.grid(True, color='white', linewidth=0.8)

ax.annotate('Cut-in',  xy=(u_cutin,  0),    xytext=(u_cutin,  40),  fontsize=12)
ax.annotate('Rated',   xy=(u_rated,  p_rated), xytext=(u_rated-1, p_rated+35), fontsize=12)
ax.annotate('Cut-out', xy=(u_cutout, p_rated), xytext=(u_cutout-1, p_rated+50),
            fontsize=12, va='center')

counts, bin_edges, patches = ax2.hist(coopers_gap.data['WS50M']+3.0,bins=100,density=True,alpha=0.25,label='data')
ax2.set_yticks([])
fig.legend(loc='upper right',bbox_to_anchor=(0.95,0.95))

plt.tight_layout()
plt.savefig('power_curve.pdf')
plt.show()

## $C_{p,max}$

In [ ]:
from enn554.wind import cp_max
lams = np.linspace(0.25,25.0,1000)
Cp_max = [cp_max(λ)['Cp_max'] for λ in lams]
Cp_betz = 16.0/27.0

fig,ax = plt.subplots()
ax.plot(lams,Cp_max,label='Cp_max with swirling')
ax.axhline(y=Cp_betz,ls=':',color='red',label='Betz limit')
ax.set_xlabel(r'Tip speed ratio, $\lambda$')
ax.set_ylabel(r'Power coefficient, $C_{p,max}$')
ax.legend()

## Betz optimium blade

In [ ]:
from enn554.wind import betz_blade
import pandas as pd
result = betz_blade(7.0,7.0,1.0,r=np.arange(0.1,1.1,0.1))
df = pd.DataFrame(result).set_index('r/R')
df.round(3)

In [ ]:
print(df.to_latex())